# HMM (Hidden Markov Model) smoothing (forward-backward)

_Artificial Intelligence — more_

**Use the future as well as the past. Combine both directions for the best estimate of each state.**

Filtering an HMM uses only past clues. But to estimate a past hidden state, the clues that came after it also help.

     The forward-backward algorithm runs two passes. The forward pass $\alpha$ gathers evidence from the start up to step $i$; the backward pass $\beta$ gathers evidence from the end back to step $i$.

---

This notebook is a practice scaffold for the **HMM (Hidden Markov Model) smoothing (forward-backward)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — Python + NumPy

In [ ]:
import numpy as np

# 2 hidden states. pi = start, T[i,j] = P(state j | state i), E[:,o] = P(obs o | state).
pi = np.array([0.6, 0.4])
T  = np.array([[0.7, 0.3],
               [0.4, 0.6]])
E  = np.array([[0.9, 0.1],     # state 0 emission probs for obs {0,1}
               [0.2, 0.8]])
obs = [0, 1, 0]                # observed sequence
N = len(obs)

alpha = np.zeros((N, 2)); beta = np.zeros((N, 2))
alpha[0] = pi * E[:, obs[0]]                       # forward init
for i in range(1, N):
    alpha[i] = (alpha[i-1] @ T) * E[:, obs[i]]
beta[N-1] = 1.0                                    # backward init
for i in range(N-2, -1, -1):
    beta[i] = T @ (E[:, obs[i+1]] * beta[i+1])

post = alpha * beta
post /= post.sum(axis=1, keepdims=True)            # smoothed P(H_i | E)
print("smoothed posteriors per step:")
print(np.round(post, 3))

## Visualize the data & results

_Umbrella world: a guard sees the boss carry an umbrella on days 1,2,4,5 but not day 3 — what is the smoothed belief that it rained each day?_

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Umbrella-world HMM (Russell & Norvig). States: [no rain, rain].
pi = np.array([0.5, 0.5])
T  = np.array([[0.7, 0.3], [0.3, 0.7]])      # weather persists day to day
E  = np.array([[0.8, 0.2],                   # no rain: umbrella unlikely
               [0.1, 0.9]])                  # rain:    umbrella likely
obs = [1, 1, 0, 1, 1]                         # umbrella on days 1,2,4,5; none on day 3
n = len(obs)

alpha = np.zeros((n, 2)); beta = np.zeros((n, 2))
alpha[0] = pi * E[:, obs[0]]
for i in range(1, n):
    alpha[i] = (alpha[i-1] @ T) * E[:, obs[i]]
beta[n-1] = 1.0
for i in range(n-2, -1, -1):
    beta[i] = T @ (E[:, obs[i+1]] * beta[i+1])

post = alpha * beta
post /= post.sum(axis=1, keepdims=True)       # smoothed P(Rain) per day
days = [1, 2, 3, 4, 5]

fig, ax = plt.subplots()
ax.plot(days, post[:, 1], "-o", color="#4ea1ff", label="P(Rain)")
ax.set_xticks(days)
ax.set_xlabel("day"); ax.set_ylabel("P(Rain = True | all evidence)")
ax.set_title("Smoothed P(Rain | all 5 umbrella observations) per day")
ax.legend()
plt.show()